# Sea level projections

* **Products used:** [DE Africa Coastlines](https://docs.digitalearthafrica.org/en/latest/data_specs/Coastlines_specs.html)



## Background

Sea level projections, along with coastline change measurements, contribute to our understanding of potential future changes in coastal areas. These projections consider various emission scenarios, providing a scientific foundation for policy decisions.

## Description

This notebook demonstrates how to access global sea level projections and visualise the data over an African country.

Datasets from the following publications will be used:
* Kopp, R.E., DeConto, R.M., Bader, D.A., Hay, C.C., Horton, R.M., Kulp, S., Oppenheimer, M., Pollard, D. and Strauss, B.H. (2017), Evolving Understanding of Antarctic Ice-Sheet Physics and Ambiguity in Probabilistic Sea-Level Projections. Earth's Future, 5: 1217-1233. https://doi.org/10.1002/2017EF000663
* Muis, S., Verlaan, M., Winsemius, H. et al. A global reanalysis of storm surges and extreme sea levels. Nat Commun 7, 11969 (2016). https://doi.org/10.1038/ncomms11969

## Getting started

To run this analysis, run all the cells in the notebook, starting with the "Load packages" cell.

### Load packages
Import Python packages that are used for the analysis.

In [1]:
%matplotlib inline
import os
import geopandas as gpd
import pandas as pd
import rioxarray as rxr
import numpy as np
from matplotlib import pyplot as plt

import datacube
from deafrica_tools.spatial import xr_rasterize, xr_vectorize


## Retrieve coastlines rates of change over a country of interest

The DE Africa Coastlines is a continental coastline monitoring service that can be accessed directly from AWS S3 bucket or OGC Web Services.
The [Coastlines dataset notebook](../../Datasets/Coastlines.ipynb) demonstrates loading DE Africa Coastlines over a small area through the Web Feature Service (WFS). 
For country scale analysis in this notebook, we will access data directly from AWS S3.

We will retrieve the `rates_of_change` layer from the geopackage hosted in AWS and load the data into a `geopandas.GeoDataFrame`.
To save memory, we will extract data over a specified country using the `country` attribute in the dataset.
This attribute was set based on intersection of a coastal location with the exclusive economic zone boundaries defined in https://deafrica-input-datasets.s3.af-south-1.amazonaws.com/deafrica-coastlines/countries_eez_deafrica.geojson
Therefore, we have to pick a country name from the names used in this boundary file. 

> If a different administrative boundary definition is desired, the entire continental dataset can be loaded and a spatial join with a user defined boundary geometry can be run.

In [2]:
# list of country names used
countries_gdf = gpd.read_file("https://deafrica-input-datasets.s3.af-south-1.amazonaws.com/deafrica-coastlines/countries_eez_deafrica.geojson")
countries = list(countries_gdf["TERRITORY1"].values)
print(countries)

['Nigeria', 'Seychelles', 'Gabon', 'Sao Tome and Principe', 'Senegal', 'Guinea-Bissau', 'Mauritania', 'Sudan', 'Togo', 'Cape Verde', 'Kenya', 'Angola', 'Federal Republic of Somalia', 'Algeria', 'South Africa', 'Djibouti', 'Ivory Coast', 'Tanzania', 'Sierra Leone', 'Tunisia', 'Republic of Mauritius', 'Eritrea', 'Morocco', 'Comores', 'Guinea', 'Mozambique', 'Liberia', 'Republic of the Congo', 'Benin', 'Namibia', 'Egypt', 'Madagascar', 'Gambia', 'Ghana', 'Libya', 'Cameroon', 'Democratic Republic of the Congo', 'Equatorial Guinea']


As an example, `Nigeria` will be used. For a different country, the name can be copied from the list above.

In [3]:
country = "Nigeria"

The following step takes a few minutes to run because the continental dataset is retrieved first before a country subset is extracted and saved in the `coaslines` variable. 

In [4]:
%%time
rates_of_change = gpd.read_file("https://deafrica-services.s3.af-south-1.amazonaws.com/coastlines/v0.4.0/deafricacoastlines_v0.4.0.gpkg", 
                           layer="rates_of_change").query(f"country == '{country}'")

CPU times: user 18min 10s, sys: 11.3 s, total: 18min 22s
Wall time: 20min 30s


The Coastlines `rates_of_change` dataset includes a list of attributes, for which detailed explanations can be found in the relevant section in the [User Guide](https://docs.digitalearthafrica.org/en/latest/data_specs/Coastlines_specs.html#Rate-of-Change-Statistics).

In [5]:
rates_of_change.head()

,uid,rate_time,sig_time,se_time,outl_time,dist_2000,dist_2001,dist_2002,dist_2003,dist_2004,...,angle_std,valid_obs,valid_span,sce,nsm,max_year,min_year,country,certainty,geometry
1396995,s11urfvn7h,-0.26,0.210,0.20,2007,-4.80,-5.79,-7.23,-8.36,-4.38,...,4,21,22,27.05,4.80,2006,2018,Nigeria,good,POINT (270982.704 813330.824)
1396996,s11urfvq96,-0.40,0.110,0.23,2007,-3.07,-3.78,-6.30,-10.09,-2.93,...,4,21,22,29.83,3.07,2006,2018,Nigeria,good,POINT (271008.705 813334.768)
1396997,s11urfvqxe,-0.38,0.102,0.22,2007,-7.17,-7.48,-7.50,-7.47,-6.98,...,6,21,22,26.78,7.17,2006,2019,Nigeria,good,POINT (271034.836 813335.068)
1396998,s11urfvxjd,-0.33,0.053,0.16,2007,-4.60,-4.60,4.20,-3.34,-5.27,...,6,21,22,18.20,4.60,2006,2019,Nigeria,good,POINT (271059.236 813345.758)
1396999,s11urfvz7x,-0.25,0.023,0.10,2007,1.17,2.69,0.48,-0.97,-2.32,...,5,21,22,11.78,-1.17,2006,2018,Nigeria,good,POINT (271084.409 813354.535)


## Sea level projection

We will first examine the gridded sea level projections provided as supporting information in Kopp et al. 2017.

This dataset provides global-mean sea-level (GMSL) and relative sea-level (RSL) projections at decadal intervals from 2010 to 2300, based on method developed in [Kopp et al. (2014)](https://doi.org/10.1002/2014EF000239).

Three Representative Concentration Pathway (RCP) scenarios were considered, RCP2.6, RCP4.5 and PCP8.5, resprenting low to high greenhouse gas emission pathways.

Relative sea level rise in centimeters are provided for a number of locations and along the gridded global coastlines at 1, 5,16.7, 50, 83.3, 95, 99, 99.5 and 99.9 percentiles of simulation frequency distributions.

In [6]:
slr = pd.read_csv('slr_data/Kopp2017/eft2271-sup-0006-2017ef000663-ds05.tsv', sep='\t', skiprows=1)

FileNotFoundError: [Errno 2] No such file or directory: 'slr_data/Kopp2017/eft2271-sup-0006-2017ef000663-ds05.tsv'

In [ ]:
slr.head()

We extract the data for gridded coastal locations and convert it to a `geodataframe` that can be visualised on a map.

> The longitude range is updated from 0 to 360 to -180 to 180 so the map will be centered around Africa

In [ ]:
grid_slr = slr[slr["Site Name"].str.startswith("grid_")].copy()
grid_slr["Latitude"] = grid_slr["Site Name"].str.split("_").str[1].astype(float)
grid_slr["Longitude"] = grid_slr["Site Name"].str.split("_").str[2].astype(float)
grid_slr["Longitude"] = grid_slr["Longitude"].apply(lambda x: x - 360 if x > 180 else x)
grid_slr_gdf = gpd.GeoDataFrame(grid_slr, geometry=gpd.points_from_xy(grid_slr['Longitude'], grid_slr['Latitude']))

We will examine 50th percentile RSL in the low and high emission scenarios for 2100. 

In [ ]:
grid_slr_gdf[(grid_slr_gdf.Scenario=="rcp26")& (grid_slr_gdf.Year==2100)].explore(
    column = " 0.5",
    tiles = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}", 
    attr ='Imagery @2022 Landsat/Copernicus, Map data @2022 Google',
    popup=True,
    cmap='viridis',
)

In [ ]:
grid_slr_gdf[(grid_slr_gdf.Scenario=="rcp85")& (grid_slr_gdf.Year==2100)].explore(
    column = " 0.5",
    #tiles = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}", 
    #attr ='Imagery @2022 Landsat/Copernicus, Map data @2022 Google',
    popup=True,
    cmap='viridis',
)

## Extreme sea levels

GTSR (Global Tide and Surge Reanalysis) ([Muis et al. 2016](https://doi.org/10.1038/ncomms11969)) is the first global reanalysis of storm surges and extreme sea levels based on hydrodynamic modelling. GTSR covers the entire world's coastline and provides estimates of extreme sea levels with a 1-in-100 year return period.

The data can be dowload from Muis, S (2016): Global Tide and Surge Reanalysis (GTSR). Version 1. 4TU.ResearchData. dataset. https://doi.org/10.4121/uuid:aa4a6ad5-e92c-468e-841b-de07f7133786

In [ ]:
esl = gpd.read_file('slr_data/Muis2016/data.zip')

In [ ]:
esl.head()

In [ ]:
esl.explore(
    column='rp00100',
#    tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
#    attr='Imagery @2022 Landsat/Copernicus, Map data @2022 Google',
    popup=True,
    cmap='viridis',
)

In [ ]:
gadm_level1 = gpd.read_file("data/gadm41_NGA_1.json.zip")
gadm_level1 = gadm_level1.to_crs(rates_of_change.crs)

slr_country= slr.sjoin_nearest(gadm_level1.to_crs(slr.crs)[["GID_0", "NAME_0", "GID_1", "NAME_1","geometry"]], how="inner", max_distance=0.01)

In [ ]:
slr_country.head()

In [ ]:
slr.explore(
    column='rp00100',
    tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    attr='Imagery @2022 Landsat/Copernicus, Map data @2022 Google',
    popup=True,
    cmap='viridis',
    #style_kwds=dict(color='red', fillOpacity=0, weight=3)
)

***

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Africa data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Slack channel](http://slack.opendatacube.org/) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [Github](https://github.com/digitalearthafrica/deafrica-sandbox-notebooks).

**Compatible datacube version:** 

In [ ]:
print(datacube.__version__)

**Last Tested:**

In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')